## Import Liblary

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading

In [9]:
df = pd.read_csv('/content/curn_cleaned_data.csv')

In [10]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## T-Test

**Business Question:** Is there a significant difference between the balances of churning customers and those of loyal (non-churning) customers?

**H0:** There is no difference in average balance
between churning and non-churning customers.

**H1:** There is a difference in the average balance

In [11]:
churn = df[df['Exited']==1]['Balance']

non_churn = df[df['Exited']==0]['Balance']

In [12]:
t_stat, p_value = stats.ttest_ind(
    churn,
    non_churn
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 11.936240300013841
P-value: 1.2755633191525477e-32


In [13]:
mean_diff = (
    churn.mean()
    - non_churn.mean()
)

ci = stats.t.interval(
    confidence=0.95,
    df=len(churn)-1,
    loc=mean_diff,
    scale=stats.sem(churn)
)

print(ci)

(np.float64(15827.343113132569), np.float64(20899.1420036844))


In [14]:
print("Churn Mean:", churn.mean())
print("Non Churn Mean:", non_churn.mean())

Churn Mean: 91108.53933726068
Non Churn Mean: 72745.2967788522


### Key Insight
**Observation:** Customers who churn tend to
have higher account balances

**Evidence:** T-test showed statistically
significant differences in balance
between churned and retained customers
(T = 11.94, p < 0.05).

The estimated mean difference
ranges between 15.8k–20.9k.

**Business Interpretations:** The bank may be losing
high-value customers.

**Recomendation:** Prioritize retention strategy
for high-balance customers.

## Chi Square

**Business Question:** Is active membership related to churn?

**H0:** There is no correlation between active membership and churn

**H1:** There is a difference in the relationship between active membership and churn

In [15]:
cont_table = pd.crosstab(
    df['IsActiveMember'],
    df['Exited']
)

cont_table

Exited,0,1
IsActiveMember,,
0,3547,1302
1,4416,735


In [16]:
chi2, p, dof, expected = stats.chi2_contingency(
    cont_table
)

print("Chi-square:", chi2)
print("P-value:", p)
print("Degrees of Freedom:", dof)

Chi-square: 242.98534164287963
P-value: 8.785858269303703e-55
Degrees of Freedom: 1


In [17]:
pd.crosstab(
    df['IsActiveMember'],
    df['Exited'],
    normalize='index'
) * 100

Exited,0,1
IsActiveMember,,
0,73.149103,26.850897
1,85.730926,14.269074


### Key Insight
**Observation:** Inactive customers show
higher churn rates.

**Evidence:** Chi-square test indicates
a statistically significant relationship
between active membership and churn
(χ² = 242.98, p < 0.05). Inactive members churned
substantially more often. The estimated mean difference
ranges between 15.8k–20.9k.

**Business Interpretations:** Customer inactivity may act as
an early warning signal of churn risk.

**Recomendation:** Build re-engagement campaigns
for inactive customers.

## Anova

**Business Question:** Does churn vary by age group?

**H0:** There is no difference in churn rates across age groups

**H1:** There is a difference between the two

In [18]:
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[18,30,50,100],
    labels=[
        'Young',
        'Middle Age',
        'Senior'
    ]
)

In [19]:
young = df[
    df['Age_Group']=='Young'
]['Exited']

middle = df[
    df['Age_Group']=='Middle Age'
]['Exited']

senior = df[
    df['Age_Group']=='Senior'
]['Exited']

f_stat, p_value = stats.f_oneway(
    young,
    middle,
    senior
)

print("P-value:", p_value)

P-value: 1.1119932795824143e-148


In [20]:
df.groupby('Age_Group')['Exited'].mean() * 100

/tmp/ipykernel_6179/466680114.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('Age_Group')['Exited'].mean() * 100


,Exited
Age_Group,
Young,7.502569
Middle Age,19.583518
Senior,44.647105


### Key Insight
**Observation:** Senior customers show
the highest churn rate

**Evidence:** ANOVA indicates significant
differences in churn across age groups
(p < 0.05).

Senior customers recorded
44.6% churn rate,
substantially higher than
middle-age (19.6%)
and young customers (7.5%).

**Business Interpretations:** Older customers may require
different engagement strategies
or may face lower banking activity,
leading to higher churn risk.

**Recomendation:** Banks should create
senior-focused retention programs
and proactive engagement.

## Corelations Significant

In [21]:
corr = df.corr(
    numeric_only=True
)

corr['Exited'].sort_values(
    ascending=False
)

,Exited
Exited,1.000000
Age,0.285323
Balance,0.118533
EstimatedSalary,0.012097
CustomerId,-0.006248
HasCrCard,-0.007138
Tenure,-0.014001
RowNumber,-0.016571
CreditScore,-0.027094
NumOfProducts,-0.047820


In [24]:
variables = [
    'CreditScore',
    'Age',
    'Balance',
    'EstimatedSalary',
    'HasCrCard',
    'Tenure',
    'NumOfProducts',
    'IsActiveMember']
results = []
for col in variables:
    corr, p_value = stats.pearsonr(
        df[col],
        df['Exited'])
    results.append([
        col,
        round(corr, 3),
        round(p_value, 5)
    ])
corr_df = pd.DataFrame(
    results,
    columns=[
        'Variable',
        'Correlation',
        'P_Value'
    ]
)
corr_df

,Variable,Correlation,P_Value
0,CreditScore,-0.027,0.00674
1,Age,0.285,0.00000
2,Balance,0.119,0.00000
3,EstimatedSalary,0.012,0.22644
4,HasCrCard,-0.007,0.47541
5,Tenure,-0.014,0.16153
6,NumOfProducts,-0.048,0.00000
7,IsActiveMember,-0.156,0.00000


### Key Insight
Age is the variable most associated with customer churn. Age shows a statistically significant positive relationship with churn, indicating that older customers tend to exhibit higher churn probability